In [2]:
# Experiment
# - Info-Gain SoRL
# - 2 Phase switch, Phase I: baseline language modeling, Phase II: SoRL with info-gain target & reward scaling.

import torch
from sorl.gat_sim import GAT, GATConfig, BOS_TOKEN_ID # <- dream GAT with separate abs / traj memory span
torch.set_float32_matmul_precision('high')  # Enable TF32 for ~2x speedup

gat_config = GATConfig(
    vocab_sizes=[BOS_TOKEN_ID+1, 16],  # 16 abstract tokens
    n_layer=4,
    n_head=4,
    n_embd=128,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

model = GAT(gat_config)
# model = model.to("cuda")
# model = torch.compile(model)

# Idea #1. 
# - cyclic compression with various attention mask pattern

In [3]:
from data.tinystory_local import TinyStoriesDataLoader, collect_rollout_statistics, AbstractionStatistics
from data.tinystory_local import visualize_dynamics
import tiktoken 

# TinyStories Dataset + GPT2 tokenizer
# -------------------------------------
num_stories = 100
max_len = 32
K = 4

loader = TinyStoriesDataLoader(num_stories=num_stories, max_len=max_len, chunk_size=K, device=model.device)

batch_size = 8
memory_span_abs = 1792
memory_span_traj = 1792
attn_blocksize = 1792
max_iterations = 2

# ---- stat collection ---
doc_len = max_len  + (max_len - 1) // K # <-- doc len contains abstract tokens
abs_stats = AbstractionStatistics(
    n_doc=num_stories,
    n_abs=doc_len - max_len,
    abs_vocab_size=model.vocab_sizes[1],
    device=model.device
)

# --- tokenizer --- 
enc = tiktoken.get_encoding("gpt2")
eot = enc._special_tokens['<|endoftext|>']

Loading 100 stories from TinyStories train...
Loaded 100 stories, 3200 tokens total, 0.01 MB
Collected 630 unique 4-chunks


In [4]:
# ----- Memory Compression SoRL ----- 
# -> Bottleneck compression Mask + Mutual distillation loss

from sorl.dream_utils import sorl_evaluate_v2, sorl_search
from sorl.topo import orthogonalize_abs_param
from collections import defaultdict
from sorl.info import SoRLLoss_v11

# --- orthogonal initialization on abs param --- 
orthogonalize_abs_param(model, do_wte=True, do_head=True)

# --- v11: mutual distillation loss ---
loss_fn = SoRLLoss_v11(model.vocab_sizes[1], decay=0.8, target_vocab_util=0.9)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
n = 2
temperature = torch.tensor([0.0, 5.0], device=model.device)
num_steps = 400
alpha_abs = 0.1
alpha_soft_zipf = 1.0
alpha_info_gain = 10.0
alpha_md = 1.0

memory_span_abs = K 
memory_span_traj = 1792

record = defaultdict(list)
img_frames = []

for step in range(num_steps): 

    optimizer.zero_grad()

    tokens, doc_ids = loader.get_batch(batch_size)

    with torch.no_grad(): 
        best_data, best_traj_ppt, best_abs_ppt, search_adv = sorl_search(tokens, model, n=n, K=K, max_iterations=max_iterations, memory_span_abs=memory_span_abs, memory_span_traj=memory_span_traj, attn_blocksize=attn_blocksize, temperature=temperature, truncate_seq_len=False)

    # --- compute loss (mutual distillation loss) --- 
    base_traj_loss, _, base_logits = model.forward(tokens, memory_span_abs, memory_span_traj, attn_blocksize)
    base_traj_loss = base_traj_loss.mean()
    info_loss, abs_loss, zipf_bigram_loss, md_loss = loss_fn(best_data, model, base_traj_loss.detach(), base_logits, memory_span_abs, memory_span_traj, attn_blocksize)    
    loss = base_traj_loss.mean() + alpha_info_gain * info_loss + alpha_abs * abs_loss + alpha_soft_zipf * zipf_bigram_loss + alpha_md * md_loss
    
    # --- log relative info gain ---
    rel_info_gain = ((-info_loss) / base_traj_loss).detach()

    # --- optimize --- 
    loss.backward() 
    optimizer.step()

    if step % 2 == 0: 
        with torch.no_grad(): 
            temperatures_eval = torch.tensor([0.0, 10.0], device=model.device)

            val_tokens, val_adv, traj_loss, abs_loss, abs_logits, abs_tokens, avg_logit_sim = sorl_evaluate_v2(tokens, model, n=2, K=K, max_iterations=max_iterations, memory_span_abs=memory_span_abs, memory_span_traj=memory_span_traj, attn_blocksize=attn_blocksize, temperature=temperatures_eval,
                                                                     truncate_seq_len=False)
            _, _, zipf_bigram_loss, md_loss = loss_fn(val_tokens, model, base_traj_loss, base_logits, memory_span_abs, memory_span_traj, attn_blocksize)
            
            abs_stats.update(abs_logits, traj_loss, rel_info_gain, abs_tokens, doc_ids)
            record['vocab_util'].append(abs_stats.vocab_util * 100)
            record['greedy_adv'].append(val_adv.item() * 100)
            record['abs_loss'].append(abs_loss.mean().item())
            record['traj_loss'].append(traj_loss.mean().item())
            record['base_traj_loss'].append(base_traj_loss.item())
            record['bigram_rep_rate'].append(abs_stats.bigram_rep_rate)
            record['kl_soft_zipf'].append(zipf_bigram_loss.item())
            record['md_loss'].append(md_loss.item())    
            record['rel_search_info_gain'].append(rel_info_gain)

        print(f"\nstep {step} | base traj loss: {base_traj_loss.item():.2f} | cond traj loss: {traj_loss.mean().item():.2f} | rel search info gain: {rel_info_gain * 100:.2f}% | greedy adv: {val_adv.item() * 100:.2f}% | vocab util: {abs_stats.vocab_util * 100:.2f}%  | avg logit sim: {avg_logit_sim:.2f} |  bigram-zipf kl: {zipf_bigram_loss.item():.2f} | bigram rep rate: {abs_stats.bigram_rep_rate:.2f} | rel info gain (search): {rel_info_gain:.2f} | md loss: {md_loss.item():.2f}")
        
        img = visualize_dynamics(abs_stats, loader, model, enc, K, step)
        img_frames.append(img)
        break


step 0 | base traj loss: 10.19 | cond traj loss: 10.74 | rel search info gain: -6.25% | greedy adv: 0.14% | vocab util: 87.50%  | avg logit sim: 0.21 |  bigram-zipf kl: 61.88 | bigram rep rate: 0.93 | rel info gain (search): -0.06 | md loss: 0.00


In [ ]:
# For mutual distillation, it's important to check
# (a). With / without distillation, how does p(s | a, M) changes, if with distillation we observe bigger p(s | a, M) then we've proven its effectiveness
# (b). Whether the 'md_loss' (mutual distillation loss) is smaller then explicitly regularized for

# Exp1 | 400 steps | mutual distillation loss w=1.0 | full logit matching
# -- p(s): 1.3 | p(s | a, M) : 1.25 | greedy adv: 8.6% | greedy info gain: 4.6% | md loss: 2.20
# Exp1 (a) | 400 steps | mutual distillation loss w=1.0 | traj logit matching
#  

# Exp2 | 400 steps | mutual distillation loss w=0.0
# -- p(s): 1.08 | p(s | a, M) : 0.78 | greedy adv: 20.5% | greedy info gain: 28.2% | md loss: 30.0

# [Observation]. Pure mutual distillation loss degrades p(s) as well as p(s | a, M)

# Exp3 | 400 steps | context distillation loss w=1.0
# -- p(s): 2.67 | p(s | a, M): 2.53 | ----

# Exp4 | 400 steps | mutual distillation loss w=1.0 | traj logit matching 
# -- p(s): 1.26 | p(s | a, M): 1.19 | greedy adv: 10.7% | greedy info gain: 4.9% | md loss: 1.84

# [Strange] it doesn't make sense to have p(s) degrades here? 



In [7]:
# Question: 
# - How do we introduce 'memory compression' in a more principled way? What can we learn from biological systems? 
# - I believe biological system simply can't store sensor observation in raw form, yet must predict them with high accuracy
#   so, intuitively they always get a 'bottleneck mask' for next-token-prediction target

# Question #1. 
# Does such 'next-token-prediction' bottleneck achieves better 'generalization' to unseen data? 
# It's reasonable to expect such 'bottleneck' degrades route memorization, but what about generalization? 

# Question #2. 
# Does such bottleneck exhibits similar 'memory address' property? 

# Question #3. 
# Does such bottleneck helps alleviating 'interference' problem? 


In [3]:
# Cyclic training with compression & memorization stages
from sorl.dream_utils import sorl_evaluate_v2, sorl_search
from sorl.topo import orthogonalize_abs_param
from collections import defaultdict
from sorl.info import SoRLLoss_v9, SoRLLoss_v10

# --- orthogonal initialization on abs param --- 
orthogonalize_abs_param(model, do_wte=True, do_head=True)

# mem_loss_fn = SoRLLoss_v9(model.vocab_sizes[1], decay=0.8, target_vocab_util=0.9)
comp_loss_fn = SoRLLoss_v10(model.vocab_sizes[1], decay=0.8, target_vocab_util=0.9)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
n = 2
temperature = torch.tensor([0.0, 5.0], device=model.device)
num_steps = 400
alpha_abs = 0.1
alpha_soft_zipf = 1.0
alpha_info_gain = 10.0
phase = "compression"

record = defaultdict(list)
img_frames = []

# Just the most biologically plausible setting, sensor input is lost, once encoded into neuron state
memory_span_abs = K 
memory_span_traj = K

for step in range(num_steps): 

    optimizer.zero_grad()

    tokens, doc_ids = loader.get_batch(batch_size)

    with torch.no_grad(): 
        best_data, best_traj_ppt, best_abs_ppt, search_adv = sorl_search(tokens, model, n=n, K=K, max_iterations=max_iterations, memory_span_abs=memory_span_abs, memory_span_traj=memory_span_traj, attn_blocksize=attn_blocksize, temperature=temperature, truncate_seq_len=False)

    # --- compute loss --- 
    cond_traj_loss, abs_loss, zipf_bigram_loss = comp_loss_fn(best_data, model, memory_span_abs, memory_span_traj, attn_blocksize)
    loss = cond_traj_loss + alpha_abs * abs_loss + alpha_soft_zipf * zipf_bigram_loss

    # --- log relative info gain ---
    base_traj_loss = model.forward(tokens, memory_span_abs, memory_span_traj, attn_blocksize)[0].mean()
    rel_info_gain = (base_traj_loss - cond_traj_loss) / base_traj_loss

    # --- optimize --- 
    loss.backward() 
    optimizer.step()

    if step % 2 == 0: 
        with torch.no_grad(): 
            temperatures_eval = torch.tensor([0.0, 10.0], device=model.device)

            val_tokens, val_adv, traj_loss, abs_loss, abs_logits, abs_tokens, avg_logit_sim = sorl_evaluate_v2(tokens, model, n=2, K=K, max_iterations=max_iterations, memory_span_abs=memory_span_abs, memory_span_traj=memory_span_traj, attn_blocksize=attn_blocksize, temperature=temperatures_eval,
                                                                     truncate_seq_len=False)
            _, _, zipf_bigram_loss = comp_loss_fn(val_tokens, model, memory_span_abs, memory_span_traj, attn_blocksize)
     
            abs_stats.update(abs_logits, traj_loss, rel_info_gain, abs_tokens, doc_ids)
            record['vocab_util'].append(abs_stats.vocab_util * 100)
            record['greedy_adv'].append(val_adv.item() * 100)
            record['abs_loss'].append(abs_loss.mean().item())
            record['traj_loss'].append(traj_loss.mean().item())
            record['base_traj_loss'].append(base_traj_loss.item())
            record['bigram_rep_rate'].append(abs_stats.bigram_rep_rate)
            record['kl_soft_zipf'].append(zipf_bigram_loss.item())
            record['rel_search_info_gain'].append(rel_info_gain)

        print(f"\n{phase} | step {step} | base traj loss: {base_traj_loss.item():.2f} | cond traj loss: {traj_loss.mean().item():.2f} | rel search info gain: {rel_info_gain * 100:.2f}% | greedy adv: {val_adv.item() * 100:.2f}% | vocab util: {abs_stats.vocab_util * 100:.2f}%  | avg logit sim: {avg_logit_sim:.2f} |  bigram-zipf kl: {zipf_bigram_loss.item():.2f} | bigram rep rate: {abs_stats.bigram_rep_rate:.2f} | rel info gain (search): {rel_info_gain:.2f}")
        
        img = visualize_dynamics(abs_stats, loader, model, enc, K, step)
        img_frames.append(img)
        # break


compression | step 0 | base traj loss: 10.19 | cond traj loss: 10.74 | rel search info gain: -6.25% | greedy adv: 0.12% | vocab util: 81.25%  | avg logit sim: 0.21 |  bigram-zipf kl: 61.92 | bigram rep rate: 0.93 | rel info gain (search): -0.06

compression | step 2 | base traj loss: 10.11 | cond traj loss: 10.63 | rel search info gain: -6.23% | greedy adv: 0.12% | vocab util: 87.50%  | avg logit sim: 0.71 |  bigram-zipf kl: 30.31 | bigram rep rate: 0.86 | rel info gain (search): -0.06

compression | step 4 | base traj loss: 9.95 | cond traj loss: 10.43 | rel search info gain: -6.29% | greedy adv: 0.13% | vocab util: 87.50%  | avg logit sim: 0.84 |  bigram-zipf kl: 15.11 | bigram rep rate: 0.80 | rel info gain (search): -0.06

compression | step 6 | base traj loss: 9.72 | cond traj loss: 10.14 | rel search info gain: -6.19% | greedy adv: 0.16% | vocab util: 87.50%  | avg logit sim: 0.90 |  bigram-zipf kl: 8.02 | bigram rep rate: 0.74 | rel info gain (search): -0.06

compression | step

In [ ]:
from sorl.stat import materialize_attention_mask_dream, AttentionMaskVisualizer
memory_span_abs = 256

data = best_data[:, :32]
# data = tokens[:, :32]

data[0,0] = 1
data = data.repeat(1, 56)
data[0,0] = 50256

attn_mask = materialize_attention_mask_dream(data, model, memory_span_abs, memory_span_traj, attn_blocksize)
attn_vis = AttentionMaskVisualizer(data, model, attn_mask)
attn_mask.float().sum() / (attn_mask.shape[0] * attn_mask.shape[1])
# attn_vis.plot_arcs()
# attn_vis.plot()

In [ ]:
# --- compute cost saving of attention mask ---
# from attn_mask alone we should be able to compute cost (count non-zero element)



tensor(0.1151)

In [23]:
1792 * 1792

3211264

In [ ]:
from sorl.forget import * 


In [ ]:
# Exp 1. K=4 | 400 steps no memory compression mask | 
#    Info gain starts aroud -6%, vocab util collapsed | at around step 230, info gain goes positive | then vocab util increases to 31%
#    Interestingly, it's precisely when vocabulary fully collapsed that info gain goes positive, on full scale experiment
#    we do not observe this phenomenon of vocabulary collapse, this explains why we don't have positive info gain in full-scale experiments too
# -- base traj loss: 1.22 | cond traj loss: 0.94 | greedy adv: 23% | info gain: 25% | vocab util: 31%

# Exp 2. K=4 | 200 steps memory_span_abs=K, memory_span_traj=K | 200 steps normal
#    memory compression mask without "bottleneck constraint"
#    First stage info gain: -6%, vocab drops to 12%, Second stage info gain rise to positive 3% in 30 steps, ends around 13%, vocab util increases 
# -- base traj loss: 1.08 | cond traj loss: 0.79 | greedy adv: 25% | info gain: 27% | vocab util: 37.5%

# Exp 3. K=4 | 400 steps bottleneck compression mask, memory_span_abs=K, memory_span_traj=K (no orthogonal initialization)
#    initially p(s | M) and p(s | a, M) drops together, then vocabulary collapse and p(s | a, M) rises whilst p(s | M) decreases 
#    the big issue is vocabulary collapse
# -- base traj loss: 7.00 | cond traj loss: 1.09 | greedy adv: 71.8% | info gain: 560% | vocab util: 6.25% (collapsed)

# Exp 4. K=4 | 200 steps bottlenck compression mask, memory_span_abs=K, memory_span_traj=K | 200 steps normal
#    with the 'manual collapse' of vocabulary, perhapse full-scale experiment can start (in 2nd stage) from vocab increase & pos info gain etc. etc?
# -- base traj loss: 1.28 | cond traj loss: 0.68 | greedy adv: 64% | info gain: 58% | vocab util: 6.25% (collapsed)

# Exp 5. K=4 | 200 steps detach on p(s) | 200 steps normal
#    dumb info gain phase hurts ppl, but it lifts up info gain & greedy adv significantly
# -- base traj loss: 1.57 | cond traj loss: 1.08 | greedy adv: 45% | info gain: 46% | vocab util: 25%

# Exp 6. K=4 | Cyclic (4:6 compression:memorization, period length=10 steps), bottleneck compression mask
# nightmare, doesn't work

# Exp 7. K=4 | 200 steps bottleneck mask with info-gain | 200 steps normal
# -- base traj loss: 1.51 | cond traj loss: 1.07 | greedy adv: 35% | info gain: 41% | vocab util: 31.25% 

# Exp 8. K=4 | 200 steps compression mask w/o info-gain | 200 steps normal
# -- base traj loss: 0.99 | cond traj loss: 0.57 | greedy adv: 31% | info gain: 40% | vocab util: 12.5%

# Exp 9. K=4 | Cyclic (4:6 compression mask w/o info gain : normal, period length=100 steps)
# -- base traj loss: 1.41 | cond traj loss: 1.12 | greedy adv: 18% | info gain: 26% | vocab util: 18.7%

# Exp 10. K=4 | 400 steps traj noise perturbation w/o info gain 
# -- base traj loss: 2.91 | cond traj loss: 1.31 | greedy adv: 0.32 | info gain: 155% | vocab util: 81%

# Exp 11. K=4 | 400 steps bottleneck compression mask, cond ppl target | orth init
# - up until step 150, p(s | M) > p(s | a, M) and the drop of p(s | M) despite training on p(s | a, M) is quite mysterious
# - after step 150, p(s | a, M) > p(s | M)
# -- base traj loss: 2.26 | cond traj loss: 0.79 | greedy adv: 41.5% | vocab util: 81.25% | info gain: 143.2% 


# Remark #1. The 'vocab collapse' issue is much less severe in full experiment settings, base training never collapse vocabulary on full-experiment
#            unlike small-scale local experiment which always collapse vocabulary and then recovers it, therefore Exp4. looks advantageous here. 

# Therefore, information bottleneck helps with ppl, whilst info gain over-trainig helps with building advantage only

In [4]:
from sorl.forget import compute_abs_stats_v5
from tqdm import tqdm
from collections import defaultdict

n = 5
temperature = torch.tensor([0.0] + [5.0] * (n - 1))
batch_size = 2

# Collect all stats
all_stats = defaultdict(list)

with tqdm(total=len(loader.stories), desc="Processing batches") as pbar:
    for i in range(0, len(loader.stories), batch_size):
        batch_indices = torch.arange(i, min(i + batch_size, len(loader.stories)))
        tokens, doc_ids = loader.get_specific(batch_indices)

        stat_dict = compute_abs_stats_v5(
            tokens, model, n=n, K=K, max_iterations=max_iterations,
            memory_span_abs=memory_span_abs, memory_span_traj=memory_span_traj, attn_blocksize=attn_blocksize,
            temperature=temperature, truncate_seq_len=False, pad_token=loader.pad_token
        )

        for key, val in stat_dict.items():
            all_stats[key].append(val.item() if hasattr(val, 'item') else val)

        pbar.update(len(batch_indices))
        # break

# Print averages
for key, vals in all_stats.items():
    print(f"{key}: {sum(vals) / len(vals):.6f}")

Processing batches: 100%|██████████| 100/100 [00:13<00:00,  7.38it/s]

base_traj_loss: 1.261729
greedy_traj_loss: 1.192577
search_traj_loss: 1.192040
greedy_adv: 0.107032
search_adv: 0.107159
greedy_abs_adv: 0.239497
search_abs_adv: 0.240033
greedy_info_gain: 0.049125
search_info_gain: 0.049661


In [5]:
if len(img_frames) > 0:
    img_frames[0].save(
        'tinystories_dynamics (SoRL + bottleneck compression mask).gif',
        save_all=True,
        append_images=img_frames[1:],
        duration=400,  # milliseconds per frame
        loop=0  # 0 = infinite loop
    )
    print(f"Saved GIF with {len(img_frames)} frames")

Saved GIF with 200 frames
